# ShopGuard Crawl-based Fraud Detection - Model Training

**Goal**: Train a multi-input MLP that classifies a crawled page as phishing (0) vs legitimate (1) using **all merged_v1 features except the URL string**, then export to ONNX for `ai-worker` inference.

**Inputs**: `merged_v1.parquet` rows where `source == 'phishpedia'` (10,395 rows; legitphish rows have no crawl content).

**Outputs** (saved to Drive folder `crawl_model_artifacts/`):
- `fraud_model.onnx` - trained PyTorch MLP
- `preprocessor.pkl` - sklearn ColumnTransformer (TF-IDF + scaler + one-hot)
- `feature_metadata.json` - input schema (column dtypes, vocab sizes, etc.)
- `model_metadata.json` - training info, metrics, threshold

**Architecture**:
```
features_text  -> TF-IDF (~3000)  --\
features_html  -> tag-token TF-IDF (~500)  --+-- concat (~3775)
features_css   -> property bag (~200)       /     -> StandardScaler
tabular meta   -> scaler + one-hot (~75) ---/        -> MLP 3775 -> 256 -> 64 -> 1
```

**Label convention** (matches `merged_v1.parquet`):
- `0 = phishing`
- `1 = legitimate`

**Excluded for leakage / dead-column reasons** (rationale in section 3):
- `url`, `domain` - URL-derived; URL classifier handles those
- `brands` - only populated on phishing rows; trivially leaks the label
- `whois_registrar` - empty in source CSVs
- `scan_date` - used only to derive certificate-age features, never as a raw feature
- `remote_ip_asn`, `remote_ip_isp` - high cardinality, mostly redundant with country


## 1. Environment Setup

In [ ]:
!pip install -q onnx onnxruntime onnxscript

import json
import math
from datetime import datetime
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

## 2. Load Data

Mount Drive and read `merged_v1.parquet`. Filter to `phishpedia` rows since `legitphish` rows have only url+label (no crawl content).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

PARQUET_PATH = '/content/drive/MyDrive/26S_CS350_Software_Engineering/dataset/merged_v1.parquet'

USE_COLUMNS = [
    'label', 'source',
    'features_text', 'features_html', 'features_css',
    'protocol', 'language', 'assets_downloaded',
    'remote_ip_country',
    'security_state', 'security_protocol', 'security_issuer',
    'security_valid_from', 'security_valid_to',
    'whois_domain_age',
    'whois_registry_created_at', 'whois_registry_expired_at',
    'scan_date',
]

df_full = pd.read_parquet(PARQUET_PATH, columns=USE_COLUMNS)
df = df_full[df_full['source'] == 'phishpedia'].drop(columns=['source']).reset_index(drop=True)

print(f"phishpedia rows: {len(df):,}")
print(f"label distribution:\n{df['label'].value_counts(dropna=False)}")
print(f"\nnull counts (top 10):\n{df.isna().sum().sort_values(ascending=False).head(10)}")

## 3. Feature Engineering

Convert JSON columns into space-delimited token strings so default whitespace tokenizers in sklearn vectorizers work. This decouples token extraction from sklearn pickling - the ai-worker only needs to copy these short helper functions, not import them from a pickle.

Numeric derivations:
- `cert_validity_days`     = `security_valid_to` - `security_valid_from`
- `cert_remaining_at_scan` = `security_valid_to` - `scan_date`
- `cert_age_at_scan`       = `scan_date` - `security_valid_from`
- `domain_age_days`        = `whois_domain_age` (already days, just NaN-imputed)
- `domain_age_to_expiry`   = `whois_registry_expired_at` - `scan_date`

Categorical buckets used:
- `language`, `protocol`, `security_state`, `security_protocol` (low cardinality)
- `security_issuer` and `remote_ip_country` (high cardinality - dropped to top-K via `min_frequency`)

In [ ]:
# Helper functions - identical copies will live in ai-worker for inference.

def html_to_tokens(s):
    """features_html is a JSON list of tag tokens. Return space-delimited string."""
    if not isinstance(s, str) or not s.strip():
        return ''
    try:
        tags = json.loads(s)
        if not isinstance(tags, list):
            return ''
        return ' '.join(str(t).lower() for t in tags if isinstance(t, str))
    except Exception:
        return ''


def css_to_tokens(s):
    """features_css is a JSON dict {property: [values]}. Return space-delimited property names.
    Values are intentionally dropped - presence of a property is the signal."""
    if not isinstance(s, str) or not s.strip():
        return ''
    try:
        d = json.loads(s)
        if not isinstance(d, dict):
            return ''
        return ' '.join(str(k).lower().replace(' ', '_') for k in d.keys())
    except Exception:
        return ''


def parse_iso(s):
    if not isinstance(s, str) or not s.strip():
        return pd.NaT
    try:
        return pd.to_datetime(s, errors='coerce', utc=True)
    except Exception:
        return pd.NaT


def days_between(a, b):
    if pd.isna(a) or pd.isna(b):
        return np.nan
    return (a - b).total_seconds() / 86400.0

In [ ]:
df['text_tokens'] = df['features_text'].fillna('').astype(str)
df['html_tokens'] = df['features_html'].map(html_to_tokens)
df['css_tokens']  = df['features_css'].map(css_to_tokens)

df['_scan_dt']        = df['scan_date'].map(parse_iso)
df['_cert_from_dt']   = df['security_valid_from'].map(parse_iso)
df['_cert_to_dt']     = df['security_valid_to'].map(parse_iso)
df['_whois_exp_dt']   = df['whois_registry_expired_at'].map(parse_iso)

df['cert_validity_days']     = df.apply(lambda r: days_between(r['_cert_to_dt'], r['_cert_from_dt']), axis=1)
df['cert_remaining_at_scan'] = df.apply(lambda r: days_between(r['_cert_to_dt'], r['_scan_dt']), axis=1)
df['cert_age_at_scan']       = df.apply(lambda r: days_between(r['_scan_dt'], r['_cert_from_dt']), axis=1)
df['domain_age_to_expiry']   = df.apply(lambda r: days_between(r['_whois_exp_dt'], r['_scan_dt']), axis=1)
df['domain_age_days']        = pd.to_numeric(df['whois_domain_age'], errors='coerce')
df['assets_downloaded']      = pd.to_numeric(df['assets_downloaded'], errors='coerce')
df['has_tls']                = df['security_protocol'].notna().astype(int)

NUMERIC_COLS = [
    'assets_downloaded',
    'domain_age_days',
    'cert_validity_days',
    'cert_remaining_at_scan',
    'cert_age_at_scan',
    'domain_age_to_expiry',
    'has_tls',
]

CATEGORICAL_COLS = [
    'language',
    'protocol',
    'security_state',
    'security_protocol',
    'security_issuer',
    'remote_ip_country',
]

for c in CATEGORICAL_COLS:
    df[c] = df[c].fillna('__missing__').astype(str)

print('numeric NaN counts:')
print(df[NUMERIC_COLS].isna().sum())
print('\ncategorical sample cardinalities:')
for c in CATEGORICAL_COLS:
    print(f'  {c}: {df[c].nunique()} unique')

## 4. Train/Val/Test Split

70/15/15, stratified by label.

In [ ]:
from sklearn.model_selection import train_test_split

before = len(df)
df = df.dropna(subset=['label']).reset_index(drop=True)
print(f"Dropped {before - len(df)} rows missing label. Remaining: {len(df):,}")

y = df['label'].astype(np.float32).values

feature_cols = ['text_tokens', 'html_tokens', 'css_tokens'] + NUMERIC_COLS + CATEGORICAL_COLS
X_df = df[feature_cols].copy()

X_temp, X_test, y_temp, y_test = train_test_split(
    X_df, y, test_size=0.15, stratify=y, random_state=SEED
)
X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp, test_size=0.1765, stratify=y_temp, random_state=SEED
)
print(f"Train: {len(X_train)}, Val: {len(X_val)}, Test: {len(X_test)}")
print(f"Train pos ratio (legit=1): {y_train.mean():.4f}")
print(f"Val pos ratio:   {y_val.mean():.4f}")
print(f"Test pos ratio:  {y_test.mean():.4f}")

## 5. Preprocessor (fit on train only)

One `ColumnTransformer` produces a single dense matrix. Numeric features are imputed (median) and standardized; high-cardinality categoricals use `min_frequency=20` so rare values fall into an `infrequent` bucket. Saved as `preprocessor.pkl`.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

preprocessor = ColumnTransformer(
    transformers=[
        ('text', TfidfVectorizer(
            max_features=3000, ngram_range=(1, 2), min_df=3,
            sublinear_tf=True, lowercase=True,
        ), 'text_tokens'),
        ('html', TfidfVectorizer(
            max_features=500, ngram_range=(1, 2), min_df=3,
            sublinear_tf=True, lowercase=False, token_pattern=r'\S+',
        ), 'html_tokens'),
        ('css', CountVectorizer(
            max_features=200, binary=True, min_df=3,
            lowercase=False, token_pattern=r'\S+',
        ), 'css_tokens'),
        ('num', Pipeline([
            ('imputer', SimpleImputer(strategy='median')),
            ('scaler', StandardScaler()),
        ]), NUMERIC_COLS),
        ('cat', OneHotEncoder(
            handle_unknown='infrequent_if_exist',
            min_frequency=20,
            sparse_output=True,
        ), CATEGORICAL_COLS),
    ],
    sparse_threshold=1.0,
)

Xtr_sp = preprocessor.fit_transform(X_train)
Xva_sp = preprocessor.transform(X_val)
Xte_sp = preprocessor.transform(X_test)

Xtr = np.asarray(Xtr_sp.todense(), dtype=np.float32)
Xva = np.asarray(Xva_sp.todense(), dtype=np.float32)
Xte = np.asarray(Xte_sp.todense(), dtype=np.float32)

IN_DIM = Xtr.shape[1]
print(f"Feature matrix: train={Xtr.shape}, val={Xva.shape}, test={Xte.shape}")
print(f"Total input dim: {IN_DIM}")

# Per-transformer output widths for the metadata file.
# Use get_feature_names_out() (prefixed by transformer name) so we don't have
# to call .transform() on sub-transformers (which would reject a DataFrame
# containing columns they didn't see during fit).
feat_names = preprocessor.get_feature_names_out()
dims = {}
offset = 0
for name, _, _ in preprocessor.transformers_:
    if name == 'remainder':
        continue
    width = int(sum(1 for n in feat_names if n.startswith(name + '__')))
    dims[name] = {'offset': offset, 'width': width}
    offset += width
assert offset == IN_DIM, f'branch widths sum {offset} != IN_DIM {IN_DIM}'
print('per-branch widths:', dims)


## 6. Model

MLP `IN_DIM -> 256 -> 64 -> 1` with dropout 0.3 + L2 weight decay. Output is the raw logit (sigmoid lives in `BCEWithLogitsLoss`).

In [ ]:
class CrawlMLP(nn.Module):
    def __init__(self, in_dim, hidden1=256, hidden2=64, dropout=0.3):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden1),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden1, hidden2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden2, 1),
        )

    def forward(self, x):
        return self.net(x)

model = CrawlMLP(in_dim=IN_DIM).to(device)
print(model)
total_params = sum(p.numel() for p in model.parameters())
print(f"\nTotal params: {total_params:,}")

## 7. Training

- Loss: `BCEWithLogitsLoss` with `pos_weight` (class imbalance correction)
- Optimizer: AdamW, lr=1e-3, weight_decay=1e-4
- Batch size 128, max 40 epochs, early stop after 6 epochs without val_loss improvement

In [ ]:
Xtr_t = torch.from_numpy(Xtr)
ytr_t = torch.from_numpy(y_train).unsqueeze(1)
Xva_t = torch.from_numpy(Xva).to(device)
yva_t = torch.from_numpy(y_val).unsqueeze(1).to(device)

train_loader = DataLoader(
    TensorDataset(Xtr_t, ytr_t),
    batch_size=128, shuffle=True,
)

pos = y_train.sum()
neg = len(y_train) - pos
pos_weight = torch.tensor([neg / pos], device=device)
print(f"pos_weight (neg/pos): {pos_weight.item():.4f}")

criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)

EPOCHS = 40
PATIENCE = 6
best_val_loss = float('inf')
patience_counter = 0
best_state = None

for epoch in range(1, EPOCHS + 1):
    model.train()
    train_loss = 0.0
    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        logits = model(xb)
        loss = criterion(logits, yb)
        loss.backward()
        optimizer.step()
        train_loss += loss.item() * xb.size(0)
    train_loss /= len(train_loader.dataset)

    model.eval()
    with torch.no_grad():
        val_logits = model(Xva_t)
        val_loss = criterion(val_logits, yva_t).item()
        val_preds = (torch.sigmoid(val_logits) > 0.5).float()
        val_acc = (val_preds == yva_t).float().mean().item()

    print(f"Epoch {epoch:02d} | train_loss={train_loss:.4f} | val_loss={val_loss:.4f} | val_acc={val_acc:.4f}")

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        patience_counter = 0
    else:
        patience_counter += 1
        if patience_counter >= PATIENCE:
            print(f"Early stopping at epoch {epoch}")
            break

model.load_state_dict(best_state)
model.eval()
print(f"\nBest val_loss: {best_val_loss:.4f}")

## 8. Evaluation

Phishing detection cares about **recall on the phishing class (label=0)** above all - missing a real scam costs more than warning on a legitimate site. Report PR curve to allow threshold tuning.

In [ ]:
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score,
    confusion_matrix, classification_report, precision_recall_curve,
)

Xte_t = torch.from_numpy(Xte).to(device)
with torch.no_grad():
    test_logits = model(Xte_t).cpu().numpy().flatten()
    test_probs = 1.0 / (1.0 + np.exp(-test_logits))

THRESHOLD = 0.5
test_preds = (test_probs > THRESHOLD).astype(np.int32)
y_test_int = y_test.astype(np.int32)

print(f"Threshold (P_legit > 0.5): {THRESHOLD}")
print(f"Accuracy:  {accuracy_score(y_test_int, test_preds):.4f}")
print(f"Precision: {precision_score(y_test_int, test_preds):.4f}")
print(f"Recall:    {recall_score(y_test_int, test_preds):.4f}")
print(f"F1:        {f1_score(y_test_int, test_preds):.4f}")
print(f"ROC-AUC:   {roc_auc_score(y_test_int, test_probs):.4f}")
print(f"PR-AUC:    {average_precision_score(y_test_int, test_probs):.4f}")

print(f"\nConfusion matrix [[TN FP][FN TP]]:\n{confusion_matrix(y_test_int, test_preds)}")
print(f"\nClassification report:\n{classification_report(y_test_int, test_preds, target_names=['phishing', 'legitimate'])}")

# PR-curve based threshold suggestion (maximises F1)
precisions, recalls, thresholds = precision_recall_curve(y_test_int, test_probs)
f1s = 2 * precisions * recalls / (precisions + recalls + 1e-9)
best_idx = np.argmax(f1s[:-1])
suggested_threshold = float(thresholds[best_idx])
print(f"\nF1-optimal threshold suggestion: {suggested_threshold:.4f} (F1={f1s[best_idx]:.4f})")

## 9. Export Artifacts

Save the model + preprocessor + schema info to Drive.

In [ ]:
import os

OUTPUT_DIR = '/content/drive/MyDrive/26S_CS350_Software_Engineering/crawl_model_artifacts'
os.makedirs(OUTPUT_DIR, exist_ok=True)

# 1. ONNX model
model.eval()
model_cpu = model.to('cpu')
dummy_input = torch.randn(1, IN_DIM, dtype=torch.float32)

onnx_path = os.path.join(OUTPUT_DIR, 'fraud_model.onnx')
torch.onnx.export(
    model_cpu,
    dummy_input,
    onnx_path,
    export_params=True,
    opset_version=17,
    do_constant_folding=True,
    input_names=['features'],
    output_names=['logit'],
    dynamic_axes={'features': {0: 'batch_size'}, 'logit': {0: 'batch_size'}},
)
print(f"OK ONNX saved: {onnx_path}")

# 2. Preprocessor (single ColumnTransformer)
preprocessor_path = os.path.join(OUTPUT_DIR, 'preprocessor.pkl')
joblib.dump(preprocessor, preprocessor_path)
print(f"OK Preprocessor saved: {preprocessor_path}")

# 3. Feature metadata - schema for ai-worker to construct the input DataFrame
feature_metadata = {
    'input_dim': int(IN_DIM),
    'branch_widths': dims,
    'columns_required': feature_cols,
    'numeric_cols': NUMERIC_COLS,
    'categorical_cols': CATEGORICAL_COLS,
    'text_col': 'text_tokens',
    'html_col': 'html_tokens',
    'css_col': 'css_tokens',
    'preprocessing_notes': {
        'html_tokens': 'JSON list of tag names -> space-delimited lowercase string (html_to_tokens)',
        'css_tokens':  'JSON dict {prop: [vals]} -> space-delimited lowercase property names with spaces->underscore (css_to_tokens)',
        'text_tokens': 'features_text as-is (fillna empty string)',
        'categorical_missing': 'fill NaN with literal "__missing__" before transform',
        'derived_numeric': {
            'cert_validity_days':     'days(security_valid_to - security_valid_from)',
            'cert_remaining_at_scan': 'days(security_valid_to - scan_date)',
            'cert_age_at_scan':       'days(scan_date - security_valid_from)',
            'domain_age_days':        'whois_domain_age (numeric coerce)',
            'domain_age_to_expiry':   'days(whois_registry_expired_at - scan_date)',
            'has_tls':                '1 if security_protocol not null else 0',
        },
    },
}
feature_metadata_path = os.path.join(OUTPUT_DIR, 'feature_metadata.json')
with open(feature_metadata_path, 'w') as f:
    json.dump(feature_metadata, f, indent=2)
print(f"OK Feature metadata saved: {feature_metadata_path}")

# 4. Training metadata
model_metadata = {
    'trained_at': datetime.now().isoformat(),
    'framework': f'PyTorch {torch.__version__}',
    'model_arch': f'MLP ({IN_DIM} -> 256 -> 64 -> 1)',
    'training_dataset': 'merged_v1.parquet (phishpedia rows only)',
    'label_convention': {'0': 'phishing', '1': 'legitimate'},
    'n_train': int(len(Xtr)),
    'n_val': int(len(Xva)),
    'n_test': int(len(Xte)),
    'threshold': float(THRESHOLD),
    'suggested_f1_threshold': suggested_threshold,
    'metrics': {
        'accuracy': float(accuracy_score(y_test_int, test_preds)),
        'precision': float(precision_score(y_test_int, test_preds)),
        'recall': float(recall_score(y_test_int, test_preds)),
        'f1': float(f1_score(y_test_int, test_preds)),
        'roc_auc': float(roc_auc_score(y_test_int, test_probs)),
        'pr_auc': float(average_precision_score(y_test_int, test_probs)),
    },
}
model_metadata_path = os.path.join(OUTPUT_DIR, 'model_metadata.json')
with open(model_metadata_path, 'w') as f:
    json.dump(model_metadata, f, indent=2, default=str)
print(f"OK Model metadata saved: {model_metadata_path}")
print(f"\nMetadata:\n{json.dumps(model_metadata, indent=2, default=str)}")

## 10. Sanity Check

Reload ONNX + preprocessor and confirm outputs match PyTorch on a random batch. Must pass before shipping to ai-worker.

In [ ]:
import onnxruntime as ort

ort_session = ort.InferenceSession(onnx_path, providers=['CPUExecutionProvider'])
reloaded_pre = joblib.load(preprocessor_path)

sample_rows = X_test.sample(n=10, random_state=SEED)
sample_dense = np.asarray(reloaded_pre.transform(sample_rows).todense(), dtype=np.float32)

with torch.no_grad():
    torch_logits = model_cpu(torch.from_numpy(sample_dense)).numpy()

ort_logits = ort_session.run(None, {'features': sample_dense})[0]

print('PyTorch logits:', torch_logits.flatten())
print('ONNX logits:   ', ort_logits.flatten())
print(f"Max abs diff: {np.abs(torch_logits - ort_logits).max():.2e}")

assert np.allclose(torch_logits, ort_logits, atol=1e-5), 'ONNX and PyTorch outputs differ!'
print('\nOK Sanity check passed. Artifacts safe to ship to ai-worker.')

## 11. ai-worker Integration Notes

Copy the 4 files from Drive into `ai-worker/worker/pipeline/crawl_model_artifacts/`:
- `fraud_model.onnx`
- `preprocessor.pkl`
- `feature_metadata.json`
- `model_metadata.json`

Suggested inference module `ai-worker/worker/pipeline/crawl_classifier.py`:

```python
import json, joblib, math
from pathlib import Path
import numpy as np
import pandas as pd
import onnxruntime as ort

_ART = Path(__file__).resolve().parent / 'crawl_model_artifacts'
_PRE  = joblib.load(_ART / 'preprocessor.pkl')
_SESS = ort.InferenceSession(str(_ART / 'fraud_model.onnx'))
_META = json.loads((_ART / 'feature_metadata.json').read_text())

# Copy of html_to_tokens / css_to_tokens / parse_iso / days_between from the notebook.
# ... (paste from training notebook section 3)

def _row_from_crawl(crawl_result) -> pd.DataFrame:
    # Build the same one-row DataFrame that ColumnTransformer was fit on.
    # crawl_result is the ai-worker CrawlResult schema.
    row = {
        'text_tokens': crawl_result.text or '',
        'html_tokens': html_to_tokens(crawl_result.html_tags_json),
        'css_tokens':  css_to_tokens(crawl_result.css_props_json),
        # numeric + categorical fields populated from crawl_result + WHOIS + TLS lookups
        ...,
    }
    return pd.DataFrame([row])

def compute_crawl_risk_score(crawl_result) -> float:
    X = _PRE.transform(_row_from_crawl(crawl_result))
    X_dense = np.asarray(X.todense(), dtype=np.float32)
    logit = float(_SESS.run(None, {'features': X_dense})[0].ravel()[0])
    p_legit = 1.0 / (1.0 + math.exp(-logit))
    return (1.0 - p_legit) * 100.0   # 0-100 fraud score, same shape as url_classifier
```

**Score direction**: matches `url_classifier`. Model emits logit -> sigmoid = P(legitimate). Fraud score = `(1 - P_legit) * 100`.

**Crawler dependency**: this classifier only works once the crawler (TBD-2) populates `features_text`, `features_html`, `features_css`, and the WHOIS / TLS / IP / language fields the same way the training dataset produced them. Until then, falling back to URL-only scoring is correct.

**Distribution caveat**: trained on phishpedia (Microsoft / UPS / DHL impersonation). E-commerce scam pages are out of distribution - validate separately before promoting this score above a low blend weight.